# Scanpy expression plots with GenomeSpy

Compare marker expression across annotated cell groups using Scanpy’s packaged `pbmc68k_reduced()` dataset. Scanpy owns the analysis; `genome_spy.integrations.scanpy` renders supported plotting calls. This workflow adapts the [core plotting tutorial](https://scanpy.scverse.org/en/stable/tutorials/plotting/core.html). Install this project with its `scanpy` extra and select that environment as the notebook kernel. The preview targets Scanpy 1.11.5.

This workflow compares dotplot, matrixplot, individual-cell heatmap and supplied matrix views; see the [roadmap](../PLAN.md) for remaining options. For embeddings, linked brushing and UMAP customization, use the [UMAP notebook](scanpy_umap.ipynb).

Compute the group dendrogram once in Python with `sc.tl.dendrogram`. It uses this dataset’s existing `X_pca` coordinates. All three plotting calls reuse the stored linkage and leaf order; GenomeSpy only renders the branches. UMAP, PCA and differential expression are not recomputed.


In [ ]:
import scanpy as sc

import genome_spy.integrations.scanpy as gsc

pbmc = sc.datasets.pbmc68k_reduced()
sc.tl.dendrogram(pbmc, groupby="bulk_labels")
pbmc

## Dotplot: summarize marker expression

Use the core tutorial’s marker dictionary and the dataset’s existing `bulk_labels`. Dot color shows the mean expression per group; dot area shows the fraction of cells with expression strictly above zero. This call uses `raw.X` by default, as in Scanpy. Hover a dot to inspect its gene, group, mean expression, and expressing-cell percentage. Set `hover=False` to disable tooltips. Summaries are calculated in Python from selected genes; GenomeSpy renders the chart and its guides.

The dendrogram orders the cell groups by their existing PCA profiles. Marker blocks retain their original order because their labels differ from the cell-group labels. The plotting call reads and validates the stored tree without changing it.


In [ ]:
marker_genes = {
    "B-cell": ["CD79A", "MS4A1"],
    "Dendritic": ["FCER1A", "CST3"],
    "Monocytes": ["FCGR3A"],
    "NK": ["GNLY", "NKG7"],
    "Other": ["IGLL1"],
    "Plasma": ["IGJ"],
    "T-cell": ["CD3D"],
}
dotplot = gsc.pl.dotplot(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    width=400,
    height=350,
    show=False,
)
dotplot.display(inline=True, controls=False)

## Matrixplot: compare scaled group means

Reuse the same PBMC data, marker dictionary and `bulk_labels`. This adapts the core tutorial’s matrixplot: Python computes each group’s mean expression, then `standard_scale="var"` scales each gene’s means from 0 to 1. Constant genes become zero. Tile color shows that scaled mean; it does not show the fraction of expressing cells.

Hover a tile to see its gene, group and scaled mean. The same stored dendrogram gives both plots identical cell-group order. No clustering or embedding is recomputed by the plotting call.


In [ ]:
matrixplot = gsc.pl.matrixplot(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    standard_scale="var",
    cmap="Blues",
    colorbar_title="column scaled\nexpression",
    width=400,
    height=350,
    show=False,
)
matrixplot.display(inline=True, controls=False)

## Heatmap: inspect individual cells

Reuse the same markers and stored dendrogram. Each row now represents one cell rather than a group mean. Cells are ordered by their existing cell-group labels and the tree’s leaf order; the colored strip shows group membership. Gene brackets retain the marker dictionary’s order.

`standard_scale="var"` scales each gene’s individual-cell values from 0 to 1 before sorting. These are different values from matrixplot’s scaled group means. Hover a tile to see the original cell ID, gene, group and scaled expression. No cells are sampled or averaged, and no clustering is rerun.
Use the mouse wheel to zoom both axes and drag to pan. Category strips and dendrograms follow the visible cells; labels outside the visible area are hidden.


In [ ]:
heatmap = gsc.pl.heatmap(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    standard_scale="var",
    cmap="viridis",
    width=400,
    height=450,
    zoom=True,
    show=False,
)
heatmap.display(inline=True, controls=False)

## A supplied numeric matrix

`matrix` displays values you already have; it does not read AnnData or compute summaries. Here it displays the group correlations already computed by `sc.tl.dendrogram` above, in the original category order. Unlike `correlation_matrix`, this generic helper does not attach a tree.


In [ ]:
group_labels = pbmc.obs["bulk_labels"].cat.categories.tolist()
matrix_values = pbmc.uns["dendrogram_bulk_labels"]["correlation_matrix"]
matrix = gsc.pl.matrix(
    matrix_values,
    xticks=group_labels,
    yticks=group_labels,
    xlabel="Cell group",
    ylabel="Cell group",
    title="Stored group correlations",
    color_map="viridis",
    width=400,
    height=350,
    show=False,
)
matrix.display(inline=True, controls=False)

**Data use and provenance**

`scanpy.datasets.pbmc68k_reduced()` loads Scanpy’s packaged, processed subset of 10x Genomics PBMC68k. No dataset copy is bundled here. The expression plots read selected genes in Python using existing annotations. Dotplot and matrixplot calculate group means; dotplot also calculates expressing-cell fractions. Heatmap keeps all 700 individual cells. Matrixplot scales each gene’s group means; heatmap scales its cell values. Scanpy computes the shared group linkage explicitly from existing PCA coordinates; GenomeSpy renders its branches. The supplied matrix displays the same stored group correlations without recomputing them.